# 04 — Analysis & Results
Aggregates all `results/predictions/` runs into a final metrics table, confusion-matrix heatmaps, and a manual error analysis.

In [ ]:
import sys, os, json, glob
from pathlib import Path
# Colab: uncomment
# PROJECT_DIR = '/content/drive/MyDrive/finllama-sentiment'
# sys.path.insert(0, PROJECT_DIR); os.chdir(PROJECT_DIR)

# Resolve the repo root regardless of the notebook's cwd (works whether launched
# from the repo root or from notebooks/, e.g. via nbconvert or Jupyter's default
# cwd-at-notebook-location behaviour). Walk upward looking for configs/experiment.yaml.
def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'configs' / 'experiment.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root (configs/experiment.yaml not found upward from %s)' % start)

ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_fpb, load_fiqa
from src.evaluation import compute_metrics
from src.utils import load_config, set_seed

cfg = load_config(str(ROOT / 'configs' / 'experiment.yaml'))
set_seed(cfg['seed'])
PRED_DIR = str(ROOT / cfg['paths']['predictions_dir'])
SUMMARY_DIR = str(ROOT / cfg['paths']['summary_dir'])
FIGURES_DIR = str(ROOT / 'presentation' / 'key_figures')
os.makedirs(SUMMARY_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid')

In [ ]:
fpb_train, fpb_test = load_fpb(
    config=cfg['datasets']['fpb']['config'],
    test_fraction=cfg['datasets']['fpb']['test_fraction'],
    seed=cfg['seed'],
)
fiqa_test = load_fiqa(neutral_band=cfg['datasets']['fiqa']['neutral_band'])
sample_lookup = {s['id']: s for s in fpb_test + fiqa_test}

## Aggregate all runs → final table

In [ ]:
def parse_run_id(run_id):
    parts = run_id.split('__')
    model = parts[0]
    ds = parts[1]
    template = parts[2] if len(parts) > 3 else '-'
    shots_str = parts[3] if len(parts) > 3 else '0shot'
    shots = int(shots_str.replace('shot', ''))
    return model, ds, template, shots

rows = []
for run_dir in sorted(glob.glob(os.path.join(PRED_DIR, '*'))):
    run_id = os.path.basename(run_dir)
    pred_file = os.path.join(run_dir, 'predictions.jsonl')
    if not os.path.exists(pred_file):
        continue

    preds = []
    with open(pred_file) as f:
        for line in f: preds.append(json.loads(line))

    samples = [sample_lookup[p['id']] for p in preds if p['id'] in sample_lookup]
    preds_matched = [p for p in preds if p['id'] in sample_lookup]
    if not samples:
        continue

    model, ds, template, shots = parse_run_id(run_id)
    m = compute_metrics(samples, preds_matched)
    rows.append({'model': model, 'dataset': ds, 'template': template, 'shots': shots,
                 'seed': cfg['seed'],
                 'accuracy': round(m['accuracy'], 4),
                 'f1_macro': round(m['f1_macro'], 4),
                 'f1_weighted': round(m['f1_weighted'], 4),
                 'coverage': round(m['coverage'], 4),
                 'n_samples': m['n_samples']})

df_results = pd.DataFrame(rows).sort_values(['dataset', 'f1_macro'], ascending=[True, False])
out_path = os.path.join(SUMMARY_DIR, 'final_table.csv')
df_results.to_csv(out_path, index=False)
print(f'Saved {len(df_results)} runs → {out_path}')
df_results

## Comparison bar chart — F1-macro

In [ ]:
# Best config per model (highest f1_macro)
best = df_results.loc[df_results.groupby(['model', 'dataset'])['f1_macro'].idxmax()]

for ds_name in ['FPB', 'FiQA']:
    subset = best[best['dataset'] == ds_name].sort_values('f1_macro', ascending=True)
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.barh(subset['model'], subset['f1_macro'], color='steelblue', edgecolor='black')
    ax.set_xlim(0, 1)
    ax.set_xlabel('F1-macro')
    ax.set_title(f'{ds_name} — Best F1-macro per model')
    for bar in bars:
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.3f}', va='center')
    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, f'f1_macro_{ds_name}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {fig_path}')

## Confusion matrices — FinLLaMA zero-shot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
label_order = ['negative', 'neutral', 'positive']

for ax, ds_name in zip(axes, ['FPB', 'FiQA']):
    run_id = f'finllama__{ds_name}__A__0shot__seed{cfg["seed"]}'
    pred_file = os.path.join(PRED_DIR, run_id, 'predictions.jsonl')
    if not os.path.exists(pred_file):
        ax.set_title(f'{ds_name} — no data'); continue

    preds = [json.loads(l) for l in open(pred_file)]
    samples = [sample_lookup[p['id']] for p in preds if p['id'] in sample_lookup]
    preds_m = [p for p in preds if p['id'] in sample_lookup]

    m = compute_metrics(samples, preds_m)
    cm = np.array(m['confusion'])
    labels = m.get('confusion_labels', label_order)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'FinLLaMA zero-shot | {ds_name}  (F1={m["f1_macro"]:.3f})')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'confusion_finllama_0shot.png'), dpi=150, bbox_inches='tight')
plt.show()

## Error analysis — sample 30 FinLLaMA errors (FPB, template A, 0-shot)

In [ ]:
import random
rng = random.Random(cfg['seed'])

run_id = f'finllama__FPB__A__0shot__seed{cfg["seed"]}'
pred_file = os.path.join(PRED_DIR, run_id, 'predictions.jsonl')

if os.path.exists(pred_file):
    preds = [json.loads(l) for l in open(pred_file)]
    errors = [
        {'text': sample_lookup[p['id']]['text'],
         'true': sample_lookup[p['id']]['label'],
         'pred': p['pred_label'],
         'raw': p['raw_output']}
        for p in preds
        if p['id'] in sample_lookup
        and p['parse_ok']
        and p['pred_label'] != sample_lookup[p['id']]['label']
    ]
    sample_errors = rng.sample(errors, min(30, len(errors)))
    print(f'{len(errors)} total errors, showing 30:')
    for i, e in enumerate(sample_errors, 1):
        print(f'\n{i:02d}. [{e["true"]} → {e["pred"]}]')
        print(f'    {e["text"][:120]}')
        print(f'    raw: "{e["raw"]}"')
else:
    print('Run notebook 03 first.')

## Error categorization (manual)
After reviewing the 30 errors above, categorize them below:

In [ ]:
# Fill in manually after reviewing the errors above.
# Keys are categories; values are counts.
error_categories = {
    'negation':           0,   # e.g. 'not profitable' → marked positive
    'sarcasm/irony':      0,
    'domain_jargon':      0,   # 'covenant breach', 'write-down'
    'numerical_reasoning': 0,  # 'shares fell 0.5%' vs 'fell 50%'
    'other':              0,
}

fig, ax = plt.subplots(figsize=(6, 3))
pd.Series(error_categories).sort_values().plot.barh(ax=ax, color='salmon', edgecolor='black')
ax.set_title('FinLLaMA error categories (FPB, 0-shot)')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'error_categories.png'), dpi=150, bbox_inches='tight')
plt.show()